# Chapter 04: Optimizers and Momentum

**Math companion:** [Beginner mathematical view](00_math_intuition.ipynb). Study its relevant sections alongside these lessons.

[Course index](../README.md) · [Previous](../03_validation_and_generalization/README.md) · [Next](../05_pytorch_model_layers/README.md)

**Goal:** Use SGD, inspect momentum, compare learning rates at fixed budgets, and load selected checkpoints.

**Study order:** read Chapter 03 first, then run this notebook from top to bottom.
This notebook contains its own setup and can run in a fresh kernel.

Before each experiment, record your prediction; afterwards, explain the output.
Related revision checkpoints: **9 (plain SGD)** in the
[revision notebook](../revision/01_training_basics_revision.ipynb).

## Contents

- [01. Replace manual updates with the SGD optimizer](#lesson-01)
- [02. Train for 10 steps using SGD](#lesson-02)
- [03. Inspect how momentum changes SGD updates](#lesson-03)
- [04. Compare momentum settings at a fixed training budget](#lesson-04)
- [05. Learning rate and momentum interact](#lesson-05)
- [06. Select and load the best validation checkpoint](#lesson-06)


## Setup

Run this cell first in a fresh kernel. It imports PyTorch and recreates the original
four examples, so this chapter does not depend on another notebook's running kernel.


In [ ]:
import torch

# Original four examples; each x is paired with the y at the same position.
x = torch.tensor([0., 1., 2., 3.])
y = torch.tensor([2., 5., 8., 11.])
print("PyTorch version:", torch.__version__)
print("Inputs:", x.tolist())
print("Targets:", y.tolist())


## Lesson 01: Replace manual updates with the SGD optimizer

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#sgd).

<a id="lesson-01"></a>


An optimizer applies parameter updates using calculated gradients. It does not replace
prediction, loss calculation, or `loss.backward()`. Plain SGD, with momentum and weight
decay set to zero, uses the update rule we already learned:

`new parameter = old parameter - learning rate * gradient`

| Manual operation | Optimizer operation |
|---|---|
| Set each parameter's `.grad` to `None` | `optimizer.zero_grad(set_to_none=True)` |
| Calculate gradients with `loss.backward()` | Still `loss.backward()` |
| Subtract learning rate times gradient from each parameter | `optimizer.step()` |

Give the optimizer the parameter tensors it should update and choose a learning rate.
`step()` updates those parameters using their stored gradients. It does not clear the
gradients; call `zero_grad()` before the next gradient calculation. For this basic SGD
configuration, the optimizer handles applying the update without tracking it for gradients.

We compare one manual update with one optimizer update. Both start at weight 1, bias 0,
use all four original examples, and have learning rate 0.05. The SGD name does not force
batch size 1: the examples used to calculate loss determine the batch size.
Run the setup cell first. **Predict:** both approaches should produce the same weight,
bias, and loss, apart from possible tiny floating-point rounding differences.

### Experiment notes

- **Prediction:** manual updates and plain SGD should agree.
- **Observation:** _Compare their gradients before the update and their final parameters and losses._
- **Explanation:** _What does `step()` replace? Which operation still calculates the gradients?_

API references: [PyTorch SGD documentation](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html)
and [zero_grad documentation](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html).
The exercise was also executed successfully with this machine's installed PyTorch.


In [ ]:
learning_rate = 0.05

# Experiment A: the manual update we already understand.
manual_w = torch.tensor(1., requires_grad=True)
manual_b = torch.tensor(0., requires_grad=True)
manual_w.grad = None
manual_b.grad = None
manual_loss = ((manual_w * x + manual_b - y) ** 2).mean()
manual_loss.backward()
print("Manual gradients:", manual_w.grad.item(), manual_b.grad.item())

with torch.no_grad():
    manual_w -= learning_rate * manual_w.grad
    manual_b -= learning_rate * manual_b.grad
    manual_after = ((manual_w * x + manual_b - y) ** 2).mean().item()

# Experiment B: identical parameters and data, using an optimizer for the update.
sgd_w = torch.tensor(1., requires_grad=True)
sgd_b = torch.tensor(0., requires_grad=True)
optimizer = torch.optim.SGD(
    [sgd_w, sgd_b], lr=learning_rate, momentum=0, weight_decay=0
)

# Clear gradients for both parameters registered with this optimizer.
optimizer.zero_grad(set_to_none=True)
sgd_loss = ((sgd_w * x + sgd_b - y) ** 2).mean()
sgd_loss.backward()  # This still calculates the gradients; no parameter update yet.
print("SGD gradients:", sgd_w.grad.item(), sgd_b.grad.item())
print("SGD parameters before step:", sgd_w.item(), sgd_b.item())

optimizer.step()  # Replaces our manual parameter-update lines.

with torch.no_grad():
    sgd_after = ((sgd_w * x + sgd_b - y) ** 2).mean().item()

print(f"Manual | weight={manual_w.item():.4f} | bias={manual_b.item():.4f} | loss={manual_after:.4f}")
print(f"SGD    | weight={sgd_w.item():.4f} | bias={sgd_b.item():.4f} | loss={sgd_after:.4f}")
print("Parameters agree:", torch.allclose(manual_w, sgd_w) and torch.allclose(manual_b, sgd_b))


## Lesson 02: Train for 10 steps using SGD

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#sgd).

<a id="lesson-02"></a>


lesson 1 showed that a single plain SGD update matches our manual update. Now repeat
10 updates, matching Chapter 01, lesson 10's original full-batch training experiment.
Run the setup cell first. This cell resets the parameters and creates a fresh optimizer.

Create the parameters and optimizer **once before the loop**. Keep the learned parameter
values between iterations. Every step clears old gradients, calculates a fresh loss and
fresh gradients, then calls `step()` to update both registered parameters.

| Operation | Purpose |
|---|---|
| `optimizer.zero_grad(set_to_none=True)` | Clear stored gradients without resetting parameters |
| `predictions = w * x + b` | Predict using the current parameters |
| `loss = ((predictions - y) ** 2).mean()` | Combine all four squared errors into one loss |
| `loss.backward()` | Calculate gradients |
| `optimizer.step()` | Apply one parameter update |

All four examples are in each batch, so each step is one complete epoch. Print loss
**after** each update. `step()` does not clear gradients automatically; the next iteration's
`zero_grad()` handles that. Plain SGD here has no momentum or weight decay.

**Predict before running:** the ten rows should match Chapter 01, lesson 10, apart from possible tiny
rounding differences. The optimizer changes how we express the update, not its mathematical rule.

### Experiment notes

- **Prediction:** plain SGD should follow the same trajectory as the manual loop.
- **Observation:** _Compare the first, second, and tenth updates with Chapter 01, lesson 10._
- **Explanation:** _Why do we clear gradients every step but initialize parameters only once?_


In [ ]:
# Initialize once; do not reset the learned parameters inside the loop.
w = torch.tensor(1., requires_grad=True)
b = torch.tensor(0., requires_grad=True)
optimizer = torch.optim.SGD([w, b], lr=0.05, momentum=0, weight_decay=0)
sgd_history = []

for step in range(1, 11):
    # Clear old gradients while preserving the learned weight and bias.
    optimizer.zero_grad(set_to_none=True)

    # Use all four examples to calculate fresh predictions, loss, and gradients.
    predictions = w * x + b
    loss = ((predictions - y) ** 2).mean()
    loss.backward()

    # Update both parameters registered with this optimizer.
    optimizer.step()

    # Measure loss AFTER the update without building a gradient-tracking graph.
    with torch.no_grad():
        updated_loss = ((w * x + b - y) ** 2).mean().item()

    sgd_history.append((step, w.item(), b.item(), updated_loss))
    print(
        f"Step {step:2d} | weight={w.item():.4f} | "
        f"bias={b.item():.4f} | loss={updated_loss:.4f}"
    )


## Lesson 03: Inspect how momentum changes SGD updates

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#momentum).

<a id="lesson-03"></a>


Plain SGD uses the current gradient for its parameter update. SGD with momentum also
keeps a running combination of earlier gradients. For the settings used here:

`stored direction = momentum * previous stored direction + current gradient`

`new parameter = old parameter - learning rate * stored direction`

On the first step, PyTorch initializes the stored direction from the first gradient.
There is no earlier history, so the two optimizers should make the same first update.
From the second step onward, momentum can change the direction used for updates.

Run the setup cell first. Compare momentum 0 and 0.9 with identical starting parameters,
learning rate 0.05, batch size 4, and 10 updates. Every printed row shows:

- `grad_w`: the weight gradient calculated **before** the update.
- `direction_w`: the direction actually used by SGD for that update, including momentum if enabled.
- `weight`, `bias`, and `loss`: values measured **after** the update.

`optimizer.zero_grad()` clears current gradients but preserves momentum state.
Create the optimizer once before the loop so its memory continues across steps.
Momentum can help maintain progress in a consistent direction, but can also carry
parameters past a good position. It does not guarantee lower loss within every budget.

### Experiment notes

- **Prediction:** the first update should match; later updates can differ because momentum remembers earlier gradients.
- **Observation:** _Compare steps 1, 2, and 3. In which run does loss rise after a good update?_
- **Explanation:** _Can the stored direction still be negative when the current gradient becomes positive? What would that do to the weight?_

Reference: [PyTorch SGD documentation](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html).
The experiment is verified against the installed PyTorch version.


In [ ]:
momentum_results = {}

for momentum in (0.0, 0.9):
    # Reset each run so momentum is the only changed setting.
    w = torch.tensor(1., requires_grad=True)
    b = torch.tensor(0., requires_grad=True)
    optimizer = torch.optim.SGD(
        [w, b], lr=0.05, momentum=momentum, weight_decay=0
    )
    history = []
    print(f"\nMomentum = {momentum}")

    for step in range(1, 11):
        # Clear current gradients; retain the optimizer's momentum memory.
        optimizer.zero_grad(set_to_none=True)
        loss = ((w * x + b - y) ** 2).mean()
        loss.backward()
        current_gradient = w.grad.item()

        optimizer.step()

        # Inspect the direction used for this update; reading it does not change it.
        if momentum == 0:
            stored_direction = current_gradient
        else:
            stored_direction = optimizer.state[w]["momentum_buffer"].item()

        with torch.no_grad():
            updated_loss = ((w * x + b - y) ** 2).mean().item()

        history.append({
            "step": step,
            "gradient": current_gradient,
            "direction": stored_direction,
            "weight": w.item(),
            "bias": b.item(),
            "loss": updated_loss,
        })
        print(
            f"Step {step:2d} | grad_w={current_gradient:8.4f} | "
            f"direction_w={stored_direction:8.4f} | "
            f"weight={w.item():.4f} | bias={b.item():.4f} | loss={updated_loss:.4f}"
        )

    momentum_results[momentum] = history


## Lesson 04: Compare momentum settings at a fixed training budget

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#momentum).

<a id="lesson-04"></a>


**Prediction before running:** momentum 0.5 should reduce overshooting and produce a
better result than 0.9 in this experiment. Test that expectation rather than treating
it as a guarantee.

Momentum controls how much of the previous stored direction survives. Learning rate
scales the resulting direction into a parameter change. A lower momentum coefficient
means less retained history; it does not guarantee a smaller update or lower final loss.
Current and previous gradients can reinforce or oppose each other.

Compare momentum 0, 0.5, and 0.9 from the same starting weight 1 and bias 0. Hold learning
rate 0.05, batch size 4, original targets, and a ten-update budget fixed. Run sections 2
and 3 first; this cell is independent of lesson 3's recorded results.

Inspect loss after **2 updates** and after **10 updates**. A setting can make faster
initial progress yet perform worse at a later budget. State the budget when saying
which setting performed better. These are training-loss comparisons; choosing settings
for unseen performance should use validation results.

### Experiment notes

- **Prediction:** momentum 0.5 will improve the ten-update result compared with 0.9.
- **Observation:** _Which setting has the lowest loss after two updates? Which after ten?_
- **Explanation:** _Why is 'lower momentum is always better' too broad a conclusion?_


In [ ]:
def run_momentum_experiment(momentum, steps=10):
    """Change only momentum; reset the parameters and optimizer for each run."""
    w = torch.tensor(1., requires_grad=True)
    b = torch.tensor(0., requires_grad=True)
    optimizer = torch.optim.SGD(
        [w, b], lr=0.05, momentum=momentum, weight_decay=0
    )
    history = []

    for step in range(1, steps + 1):
        # Preserve learned parameters and momentum memory between steps.
        optimizer.zero_grad(set_to_none=True)
        loss = ((w * x + b - y) ** 2).mean()
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            updated_loss = ((w * x + b - y) ** 2).mean().item()
        history.append({
            "step": step, "weight": w.item(),
            "bias": b.item(), "loss": updated_loss,
        })

    return history


# Identical data, initialization, learning rate, and update count in every run.
momentum_comparison = {}
for momentum in (0.0, 0.5, 0.9):
    history = run_momentum_experiment(momentum)
    momentum_comparison[momentum] = history
    print(f"\nMomentum = {momentum}")
    for step in (1, 2, 3, 5, 10):
        row = history[step - 1]
        print(
            f"Step {step:2d} | weight={row['weight']:.4f} | "
            f"bias={row['bias']:.4f} | loss={row['loss']:.4f}"
        )

print("\nSame settings, different update budgets:")
print("Momentum | Loss after 2 updates | Loss after 10 updates")
for momentum, history in momentum_comparison.items():
    print(f"{momentum:8.1f} | {history[1]['loss']:20.6f} | {history[9]['loss']:21.6f}")


## Lesson 05: Learning rate and momentum interact

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#momentum).

<a id="lesson-05"></a>


Momentum combines previous and current gradients into a stored direction. Learning rate
scales that direction into a parameter update. For the same direction −29.5, learning
rate 0.05 gives an increase of 1.475, while rate 0.01 gives an increase of 0.295.

**Prediction:** lowering learning rate may reduce the large swings observed with momentum
0.9. It does not guarantee better performance at every training budget. The first step
will be smaller because both runs start at the same gradients; later gradients and stored
directions can differ as the runs follow different parameter trajectories.

Run the setup cell first. Keep momentum 0.9, original targets, batch size 4, starting
weight 1 and bias 0, and 100 updates fixed. Change only learning rate: 0.05 versus 0.01.
Evaluate the same reserved validation examples after each update without training on them.
Every printed loss is measured after that update using the same current parameters.

Compare the results at 10 and 100 updates. Training loss describes optimization progress;
validation loss helps assess performance on reserved examples and choose settings.

### Experiment notes

- **Prediction:** smaller initial updates and possibly smaller swings at learning rate 0.01.
- **Observation:** _Which rate has lower training and validation losses at 10 updates? At 100?_
- **Explanation:** _Why are the later updates not necessarily exactly five times smaller, even though the learning rates differ by a factor of five?_


In [ ]:
learning_rate_results = {}
validation_x = torch.tensor([4., 5.])
validation_y = torch.tensor([14., 17.])

for learning_rate in (0.05, 0.01):
    # Reset both parameters and the optimizer's history for an independent run.
    w = torch.tensor(1., requires_grad=True)
    b = torch.tensor(0., requires_grad=True)
    optimizer = torch.optim.SGD(
        [w, b], lr=learning_rate, momentum=0.9, weight_decay=0
    )
    history = []
    print(f"\nLearning rate = {learning_rate}, momentum = 0.9")

    for step in range(1, 101):
        optimizer.zero_grad(set_to_none=True)
        loss = ((w * x + b - y) ** 2).mean()
        loss.backward()
        optimizer.step()

        # Measure both datasets without gradients or further parameter updates.
        with torch.no_grad():
            training_loss = ((w * x + b - y) ** 2).mean().item()
            validation_loss = (
                (w * validation_x + b - validation_y) ** 2
            ).mean().item()

        history.append({
            "step": step, "weight": w.item(), "bias": b.item(),
            "training_loss": training_loss, "validation_loss": validation_loss,
        })
        if step in (1, 2, 3, 5, 10, 20, 50, 100):
            print(
                f"Step {step:3d} | weight={w.item():.4f} | bias={b.item():.4f} | "
                f"training={training_loss:.6f} | validation={validation_loss:.6f}"
            )

    learning_rate_results[learning_rate] = history

print("\nSame momentum, different learning rates and update budgets:")
print("Rate | Train at 10 | Valid at 10 | Train at 100 | Valid at 100")
for rate, history in learning_rate_results.items():
    at_10 = history[9]
    at_100 = history[99]
    print(
        f"{rate:.2f} | {at_10['training_loss']:.6f} | {at_10['validation_loss']:.6f} | "
        f"{at_100['training_loss']:.6f} | {at_100['validation_loss']:.6f}"
    )


## Lesson 06: Select and load the best validation checkpoint

**Mathematical view:** [Related explanation and worked example](../03_validation_and_generalization/00_math_intuition.ipynb#checkpoint).

<a id="lesson-06"></a>


Select the saved parameters with the lowest validation loss, rather than automatically
using the final parameters. Run lesson 5 first; this cell reads its `learning_rate_results`
histories and does not perform any further training.

For each learning rate, find the lowest validation loss across the same 100-update budget.
Compare those two best checkpoints using validation loss, then load the selected weight
and bias into separate tensors for evaluation. These choices use validation data, not test data.

`min(history, key=lambda row: row["validation_loss"])` finds the recorded row with the
smallest validation loss. Its stored weight and bias are numeric snapshots from that
update. `.copy()` keeps a separate checkpoint record when adding configuration information.

The final row and best row can differ. Very small losses on this tiny validation set
are evidence about these two reserved examples; they do not guarantee results on all inputs.

### Experiment notes

- **Prediction:** the best checkpoint may occur before the last update.
- **Observation:** _For each rate, record the best step, best validation loss, and final validation loss._
- **Explanation:** _Why should evaluation use the saved parameters from that best step rather than the last global `w` and `b`?_


In [ ]:
rate_checkpoints = {}
print("Rate | Best step | Best validation | Final validation")

for rate, history in learning_rate_results.items():
    # Search all recorded updates using the validation-selection criterion.
    best = min(history, key=lambda row: row["validation_loss"])
    checkpoint = best.copy()
    checkpoint["learning_rate"] = rate
    checkpoint["momentum"] = 0.9
    rate_checkpoints[rate] = checkpoint
    print(
        f"{rate:.2f} | {best['step']:9d} | {best['validation_loss']:.8f} | "
        f"{history[-1]['validation_loss']:.8f}"
    )

# Choose among the best checkpoints, rather than comparing only final rows.
selected_rate = min(
    rate_checkpoints,
    key=lambda rate: rate_checkpoints[rate]["validation_loss"],
)
selected_checkpoint = rate_checkpoints[selected_rate].copy()

# Load the chosen saved numbers, preserving earlier runs and optimizer objects.
selected_w = torch.tensor(selected_checkpoint["weight"])
selected_b = torch.tensor(selected_checkpoint["bias"])
with torch.no_grad():
    selected_predictions = selected_w * validation_x + selected_b
    selected_validation_loss = (
        (selected_predictions - validation_y) ** 2
    ).mean().item()

print("\nSelected learning rate:", selected_rate)
print("Selected step:", selected_checkpoint["step"])
print("Selected weight and bias:", selected_w.item(), selected_b.item())
print("Validation predictions:", selected_predictions.tolist())
print(f"Validation loss after loading: {selected_validation_loss:.8f}")
